In [216]:
import sqlite3 as sq3
import pandas as pd

# Create your connection.
con = sq3.connect('long_backtest_data.sql')
df = pd.read_sql_query("SELECT * FROM data", con)
con.close()

result = round(df.groupby('symbol').agg(balance=('balance', 'max')),2)
result['gains'] = round(result-10000, 2)
result['performance'] =  round((result['gains']/10000) * 100, 2)
result["balance"] = result["balance"].map("${:,.0f}".format)

df_summary = result.groupby(['symbol','balance','performance'])['gains'].sum().sort_values(ascending=False)
df_summary.loc['Grand Total'] = df_summary.sum()
df_summary = df_summary.reset_index()

df_summary['gains'] = df_summary['gains'].map("${:,.0f}".format)

pd.set_option("display.max_columns", None) 
pd.set_option("display.max_rows", None) 

header = 'From: Oct-5 22 To:  Oct-5 24 <br> Inversion: $10,000 c/u <br> stop: close5minSmooth < ema25_5min <br> target: (buying_price-current_price) >= 2<br>'

df_summary.columns = pd.MultiIndex.from_product([['Backtesting'], df_summary.columns])
styler = df_summary.style.set_caption(header).set_table_styles([{
    'selector': 'caption',
    'props': [
        ('color', 'Purple'),
        ('font-size', '15px'),
        ('font-style', 'italic'),
        ('font-weight', 'bold'),
        ('text-align', 'Left')
    ]
}])

display(styler) 

In [217]:
import sqlite3 as sq3
import pandas as pd
import numpy as np

con = sq3.connect('trading_vow.sql')
df = pd.read_sql_query("SELECT * FROM data", con)
con.close()

quantity = df['Quantity']
prevPrice = df['Price'].shift(1)
currentPrice = df['Price']

priceForShort = prevPrice-currentPrice 
priceForLongs = currentPrice-prevPrice

prevSide = df['Side'].shift(1).iloc[1::2]
df['Gain Loss'] = np.where(
                        (df['Side'] == 'B') & (prevSide == 'S'), round(priceForShort * quantity,2) , 
                            np.where((df['Side'] == 'S') & (prevSide == 'B'), round(priceForLongs * quantity,2), 0))

df['Performance'] = round((df['Gain Loss']/10000) * 100, 2)
df['Performance Yearly'] = round((df['Performance']/2), 2)
df['Performance Montly'] = round((df['Performance Yearly']/12), 2)
df_summary = df.groupby(['Symbol'])[['Gain Loss','Performance','Performance Yearly','Performance Montly']].sum().sort_values('Performance', ascending=False)

df_summary.loc['Grand Total'] = df_summary.sum()
df_summary = df_summary.reset_index()

df_summary['Gain Loss'] = df_summary['Gain Loss'].map("${:,.0f}".format)
df_summary['Performance'] = df_summary['Performance'].map("{:,.2f}".format)
df_summary['Performance Yearly'] = df_summary['Performance Yearly'].map("{:,.2f}".format)
df_summary['Performance Montly'] = df_summary['Performance Montly'].map("{:,.2f}".format)

pd.set_option("display.max_columns", None) 
pd.set_option("display.max_rows", None) 

header = "From: Oct-5 22 To:  Oct-5 24 <br> Inversion: $10,000 c/u Short and Long"

styler = df_summary.style.set_caption(header).set_table_styles([{
    'selector': 'caption',
    'props': [
        ('color', 'Purple'),
        ('font-size', '15px'),
        ('font-style', 'italic'),
        ('font-weight', 'bold'),
        ('text-align', 'Left')
    ]
}])

display(styler) 

,Symbol,Gain Loss,Performance,Performance Yearly,Performance Montly
0,AMD,"$93,112",931.07,465.58,38.87
1,AMZN,"$44,256",442.60,221.30,18.46
2,GOOGL,"$39,714",397.05,198.81,16.54
3,NVDA,"$20,820",208.20,104.14,8.61
4,AAPL,"$-8,445",-84.43,-42.21,-3.56
5,Grand Total,"$189,456","1,894.49",947.62,78.92


In [218]:
import sqlite3 as sq3
import pandas as pd

con = sq3.connect('trading_vow.sql')
df = pd.read_sql_query("SELECT * FROM data", con)
con.close()

df.to_csv('trading_vow_oct_2022_oct_2024.csv', index=False)